In [ ]:
# Remove any old version of the SDK
!pip uninstall -y openai-agents

# Install the exact version used in this course
%pip install --no-cache-dir openai-agents==0.2.2

# python-dotenv reads the .env file; LangChain is used in later modules
%pip install python-dotenv langchain-openai==0.2.1

# Pydantic validates and organises data (used later for structured outputs)
%pip install --upgrade pydantic


Found existing installation: openai-agents 0.2.2
Uninstalling openai-agents-0.2.2:
  Successfully uninstalled openai-agents-0.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 6.1 MB/s eta 0:00:00


In [ ]:
import os
from agents import AsyncOpenAI, set_tracing_disabled
from IPython.display import display, Markdown

def get_gemini_key():
    # Option A: Google Colab Secrets (key icon in the left sidebar)
    try:
        from google.colab import userdata
        for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
            try:
                key = userdata.get(name)
                if key:
                    print(f"Key loaded from Colab Secrets ({name})")
                    return key
            except Exception:
                pass   # secret missing or notebook access switched off
    except ImportError:
        pass           # not running in Colab
    # Option B: .env file on your laptop
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if key:
        print("Key loaded from .env / environment variable")
    return key

gemini_api_key = get_gemini_key()
if not gemini_api_key:
    raise ValueError("Gemini key not found. Colab: add a secret named GEMINI_API_KEY ...")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(api_key = gemini_api_key, base_url = GEMINI_BASE_URL)

set_tracing_disabled(True)
print("Gemini client successfully configured.")
print(gemini_api_key[:5])


Key loaded from Colab Secrets (GEMINI_API_KEY)
Gemini client successfully configured.
AQ.Ab


In [ ]:
GEMINI_MODEL = "gemini-3.8-flash"           # smart + fast, free tier (default)
GEMINI_LITE_MODEL = "gemini-3.5-flash-lite" # fastest, for simple tasks

# OPTIONAL: list the Gemini models your key can use
models = await gemini_client.models.list()
for m in models.data:
    if "gemini" in m.id:
        print(m.id.replace("models/", ""))


gemini-2.5-flash
gemini-2.5-pro
gemini-2.5-flash-preview-tts
gemini-2.5-pro-preview-tts
gemini-flash-latest
gemini-flash-lite-latest
gemini-pro-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
gemini-3.1-pro-preview
gemini-3.1-pro-preview-customtools
gemini-3.1-flash-lite-preview
gemini-3.1-flash-lite
gemini-3-pro-image-preview
gemini-3-pro-image
gemini-3.1-flash-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-lite-image
gemini-nano-banana-2.1
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-omni-flash-preview
gemini-omni-1.1-flash
gemini-3.5-transcribe
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-4-argon
gemini-3.1-flash-tts-preview
gemini-3.8-flash-tts
gemini-3.8-flash-lite-tts
gemini-robotics-er-2-preview
gemini-2.5-computer-use-preview-10-2025
gemini-embedding-001
gemini-embedding-2-preview
gemini-embedding-2
gemini-3.5-transcribe-live
gemini-2.5-flash-native-audio-latest
gemini-2.5-flash-native-audio-preview-09-2025
gemini-2.5-flash-native

In [ ]:
def print_markdown(text):
    display(Markdown(text))

def show_usage(result, label = ""):
    u = result.context_wrapper.usage
    print(f"{label} input tokens: {u.input_tokens} | output tokens: {u.output_tokens} | total: {u.total_tokens}")


In [ ]:
from agents import Agent
market_researcher_instructions = """
Context:
You are a market research assistant helping analyze companies, industries, and competitors.

Instructions:
When given a question, provide a short factual answer based on your knowledge.

Output:
Start with a verdict prefix: either "✅ FACT:" or "❌ UNKNOWN:"
Follow with a concise one-sentence explanation.
"""

In [ ]:
from agents import Agent, Runner, SQLiteSession, OpenAIChatCompletionsModel

market_researcher_agent = Agent(name = "Market Researcher",
                                instructions = market_researcher_instructions,
                                model = OpenAIChatCompletionsModel(model = GEMINI_LITE_MODEL,
                                                                   openai_client = gemini_client))


In [ ]:
q1 = "What is the market share of Tesla in the US EV market?"
resp1 = await Runner.run(starting_agent = market_researcher_agent, input = q1)
print_markdown(f"🤖 Agent:\n{resp1.final_output}")

q2 = "How does that compare to last year?"
resp2 = await Runner.run(starting_agent = market_researcher_agent, input = q2)
print_markdown(f"🤖 Agent:\n{resp2.final_output}")


🤖 Agent:
✅ FACT: Tesla's market share in the US electric vehicle market is estimated to be around 48% to 50% as of recent 2024 industry data.

🤖 Agent:
❌ UNKNOWN: Please specify which company, metric, or time period you are referring to so I can provide a relevant comparison.

In [ ]:
# Create a session instance (the agent's conversation diary)
session = SQLiteSession("conversation")

# First interaction
user_input1 = "What is the market share of Tesla in the US EV market?"
response1 = await Runner.run(starting_agent = market_researcher_agent,
                             input = user_input1,
                             session = session)        # <- NEW
print_markdown(f"🤖 Agent:\n{response1.final_output}")

# Second interaction - same session, so the agent sees the history
user_input2 = "How does that compare to last year?"
response2 = await Runner.run(starting_agent = market_researcher_agent,
                             input = user_input2,
                             session = session)
print_markdown(f"🤖 Agent:\n{response2.final_output}")


🤖 Agent:
✅ FACT: Tesla's market share in the US electric vehicle market is estimated to be around 49% as of mid-2024, maintaining its position as the leading EV seller despite facing increased competition.

🤖 Agent:
✅ FACT: This represents a decline from the previous year, when Tesla held approximately 55% to 60% of the US EV market, as traditional automakers and newer entrants expanded their EV offerings.